# Day 1 — Option basics: payoff versus profit

Educational experiments only. Production code must never import this notebook.
No market data, pricing model, Greeks, implied volatility, or trading strategies.

Run cells from top to bottom in a fresh Python kernel. Change the example inputs
and rerun to see how expiry outcomes change.

## Vocabulary

- **Call:** the buyer has the right to buy the underlying at the strike.
- **Put:** the buyer has the right to sell the underlying at the strike.
- **Long:** bought the option and paid the premium.
- **Short:** sold the option to open a position and received the premium. If assigned,
  a short call requires delivery of shares; a short put requires buying shares.
- **Strike (K):** the contract's agreed exercise price per share.
- **Expiry:** when the contract ends. American-style exercise can occur before expiry;
  European-style exercise occurs only at expiry.
- **Premium (p):** the price of the option, quoted here in USD per share.
- **Payoff:** the option's value at expiry, before the premium.
- **Profit:** payoff minus premium for a long option. We ignore other costs here.

Long/short describes the position, not how long it lasts. Options are distinct from
owning or short-selling the underlying stock.

## Purpose, inputs, outputs, and assumptions

These experiments explain why an option can finish in the money but lose money for
its buyer. Inputs are nonnegative expiry stock price S_T, positive strike K, and
nonnegative premium p. Outputs are expiry payoff and profit in **USD per share**.
A separate example assumes a **100-share multiplier**; real contract terms must be checked.

We hold until expiry and ignore fees, financing, taxes, dividends, and early exercise.
The payoff is the economic exercise value, not a simulation of share delivery or
actual broker exercise procedures. Premiums below are hypothetical inputs, not
market observations or theoretical model prices.


## Mathematical relationships

For a long call:

- Payoff = max(S_T - K, 0)
- Profit = payoff - p
- Expiry breakeven = K + p

For a long put:

- Payoff = max(K - S_T, 0)
- Profit = payoff - p
- Expiry breakeven = K - p, if this is a feasible nonnegative stock price

For the corresponding short position, payoff is the negative of the long payoff,
and profit is premium received minus the long payoff. Long and short profits sum
to zero when they use the same premium and exclude costs.

These relationships describe expiry outcomes. They do not price an option before expiry.


In [ ]:
# Simple notebook-only definitions. Do not move these into the application on Day 1.
def call_payoff(stock_at_expiry, strike):
    return max(stock_at_expiry - strike, 0)


def put_payoff(stock_at_expiry, strike):
    return max(strike - stock_at_expiry, 0)


strike = 100.0
call_premium = 5.0
put_premium = 4.0
contract_multiplier = 100  # Assumption for these examples only.


## Worked call example

Buy a $100-strike call for $5 per share. If the stock finishes at $112, payoff is
$12 and profit is $7 per share, or $700 for our assumed 100-share contract.
At $103 the call is ITM, but its $3 payoff is less than the $5 premium: a $2 loss.
Breakeven is $105 at expiry.


In [ ]:
for stock_at_expiry in [100.0, 103.0, 105.0, 112.0]:
    payoff = call_payoff(stock_at_expiry, strike)
    profit = payoff - call_premium
    print(f"Call | stock ${stock_at_expiry:.2f} | payoff ${payoff:.2f}/share | "
          f"profit ${profit:.2f}/share | contract profit ${profit * contract_multiplier:.2f}")


## Worked put example

Buy a $100-strike put for $4 per share. If the stock finishes at $90, payoff is
$10 and profit is $6 per share, or $600 for our assumed 100-share contract.
At $98 the put is ITM but loses $2 per share after the premium.
Breakeven is $96 at expiry.


In [ ]:
for stock_at_expiry in [90.0, 96.0, 98.0, 100.0, 112.0]:
    payoff = put_payoff(stock_at_expiry, strike)
    profit = payoff - put_premium
    print(f"Put  | stock ${stock_at_expiry:.2f} | payoff ${payoff:.2f}/share | "
          f"profit ${profit:.2f}/share | contract profit ${profit * contract_multiplier:.2f}")


## Intrinsic value, time value, and moneyness

Before expiry, intrinsic value uses **today's** stock price S:
call intrinsic = max(S - K, 0), put intrinsic = max(K - S, 0).
Time (extrinsic) value = option premium - intrinsic value.
The same max expression appears in expiry payoff, but the observation time differs.
We do not estimate a premium here.

| Condition | Call | Put |
|---|---|---|
| S > K | ITM | OTM |
| S = K | ATM | ATM |
| S < K | OTM | ITM |

ATM is used exactly in these experiments; in practice it may mean near the strike.
Moneyness ignores the premium. At expiry no time value remains.


In [ ]:
current_stock = 103.0  # Before expiry; separate from stock_at_expiry.
hypothetical_call_premium = 5.0
call_intrinsic = call_payoff(current_stock, strike)
call_time_value = hypothetical_call_premium - call_intrinsic
print(f"Before expiry: call intrinsic ${call_intrinsic:.2f}, time value ${call_time_value:.2f}")

current_stock = 98.0
hypothetical_put_premium = 4.0
put_intrinsic = put_payoff(current_stock, strike)
put_time_value = hypothetical_put_premium - put_intrinsic
print(f"Before expiry: put intrinsic ${put_intrinsic:.2f}, time value ${put_time_value:.2f}")


## Experiment: long and short across expiry prices

The table shows isolated call and put positions for learning, not a strategy builder.
A long option can lose its full premium. An uncovered short call has theoretically
unlimited loss as the stock price rises. A short put can lose K - p per share when
the stock reaches zero (for these inputs). Receiving premium is not guaranteed profit.


In [ ]:
expiry_prices = [0.0, 80.0, 90.0, 96.0, 98.0, 100.0, 103.0, 105.0, 112.0, 120.0]
print("All payoff/profit values in USD per share")
print(f"{'Stock':>7} {'Call payoff':>12} {'Long call':>10} {'Short call':>11} "
      f"{'Put payoff':>11} {'Long put':>10} {'Short put':>10}")
for stock_at_expiry in expiry_prices:
    call = call_payoff(stock_at_expiry, strike)
    put = put_payoff(stock_at_expiry, strike)
    print(f"{stock_at_expiry:7.2f} {call:12.2f} {call - call_premium:10.2f} "
          f"{call_premium - call:11.2f} {put:11.2f} {put - put_premium:10.2f} "
          f"{put_premium - put:10.2f}")


## Validation: check by hand, then run assertions

Check the worked examples, at-strike and worthless expiry cases, expiry breakevens,
and the stock-at-zero boundary. Verify that payoff is nonnegative for the holder,
long profit is no less than the negative premium, and matched long/short profits cancel.
These are notebook learning checks, not production analytics tests.


In [ ]:
assert call_payoff(112, 100) == 12
assert call_payoff(112, 100) - 5 == 7
assert put_payoff(90, 100) == 10
assert put_payoff(90, 100) - 4 == 6
assert call_payoff(100, 100) == put_payoff(100, 100) == 0
assert call_payoff(80, 100) - 5 == -5
assert put_payoff(120, 100) - 4 == -4
assert call_payoff(105, 100) - 5 == 0
assert put_payoff(96, 100) - 4 == 0
assert call_payoff(103, 100) - 5 == -2  # ITM does not imply positive profit.
assert put_payoff(98, 100) - 4 == -2
assert put_payoff(0, 100) == 100
assert 4 - put_payoff(0, 100) == -96
assert 7 * contract_multiplier == 700
assert 6 * contract_multiplier == 600
for stock_at_expiry in expiry_prices:
    for payoff, premium in [(call_payoff(stock_at_expiry, strike), call_premium),
                            (put_payoff(stock_at_expiry, strike), put_premium)]:
        assert payoff >= 0
        assert payoff - premium >= -premium
        assert (payoff - premium) + (premium - payoff) == 0
print("All option-basics checks passed.")


## Try it yourself

1. Keep strike at $100 and expiry stock at $112. Change the call premium from $5 to $8.
   Which changes: payoff, profit, or both?
2. Find an ITM put that still loses money after its premium.
3. Add expiry stock prices of $150 and $200. What happens to a short call's profit?
4. Change the contract multiplier from 100 to 10. Which per-share values change?
5. Explain why these tables cannot tell you what an option should cost today.

If you edit the worked-example inputs, restore them before running the original
assertions, or update your expected results deliberately after checking by hand.

## Sources

- [SEC: An Introduction to Options](https://www.investor.gov/introduction-investing/general-resources/news-alerts/alerts-bulletins/investor-bulletins-63)
- [OIC: Options Pricing](https://prd-web.optionseducation.org/optionsoverview/options-pricing)
- [CME: Moneyness and Intrinsic Value](https://www.cmegroup.com/education/courses/introduction-to-options/calculating-options-moneyness-and-intrinsic-value)
